# SimPy 是 Python 中一个用于做“离散事件仿真”的库

所谓“离散事件仿真”，可以简单理解为：

系统状态不是连续变化的，而是在某些时间点发生事件，系统才发生变化。

比如：

- 顾客到达银行
- 顾客开始办理业务
- 顾客办理结束离开
- 机器开始加工零件
- 机器加工完成
- 车辆到达红绿灯
- 电梯到达某一层

这些事件都发生在某些具体时间点，因此非常适合用 SimPy 模拟。

它特别适合回答这样的问题：

- 如果我增加一个服务窗口，平均等待时间会不会变短？
- 如果顾客来得更频繁，系统会不会拥堵？
- 如果机器加工时间变长，生产效率会下降多少？

# SimPy 程序的基本结构

一个典型的 SimPy 程序通常是这样：
```python
import simpy

def process(env):
    # 事件1
    yield env.timeout(时间) # 表示仿真时间等待。
    # 事件2

env = simpy.Environment() # Environment 可以理解为一个“虚拟时钟”。它负责推进仿真的时间。
env.process(process(env)) # 过程就是系统中的一个活动(e.g. 一台机器从开始加工到加工完成),过程通常用 Python 的生成器函数表示，也就是带有 yield 的函数。

env.run()
```

如果涉及资源：

```python
resource = simpy.Resource(env, capacity=数量)

with resource.request() as req:
    yield req
    # 使用资源
    yield env.timeout(使用时间)
```

In [2]:
import simpy


def customer(env, name, counter, wait_times):
    arrival_time = env.now
    print(f"{env.now:>2} 分钟：{name} 到达银行")

    with counter.request() as request:
        yield request

        start_time = env.now
        wait_time = start_time - arrival_time
        wait_times.append(wait_time)

        print(f"{env.now:>2} 分钟：{name} 开始办理业务，等待了 {wait_time} 分钟")

        yield env.timeout(5)

        print(f"{env.now:>2} 分钟：{name} 办理完成并离开")


def customer_generator(env, counter, wait_times):
    for i in range(5):
        env.process(customer(env, f"顾客{i + 1}", counter, wait_times))
        yield env.timeout(2)


env = simpy.Environment()
counter = simpy.Resource(env, capacity=2)

wait_times = []

env.process(customer_generator(env, counter, wait_times))

env.run()

average_wait = sum(wait_times) / len(wait_times)
print(f"\n平均等待时间：{average_wait} 分钟")


 0 分钟：顾客1 到达银行
 0 分钟：顾客1 开始办理业务，等待了 0 分钟
 2 分钟：顾客2 到达银行
 2 分钟：顾客2 开始办理业务，等待了 0 分钟
 4 分钟：顾客3 到达银行
 5 分钟：顾客1 办理完成并离开
 5 分钟：顾客3 开始办理业务，等待了 1 分钟
 6 分钟：顾客4 到达银行
 7 分钟：顾客2 办理完成并离开
 7 分钟：顾客4 开始办理业务，等待了 1 分钟
 8 分钟：顾客5 到达银行
10 分钟：顾客3 办理完成并离开
10 分钟：顾客5 开始办理业务，等待了 2 分钟
12 分钟：顾客4 办理完成并离开
15 分钟：顾客5 办理完成并离开

平均等待时间：0.8 分钟
